In [3]:
from pathlib import Path
import ast
import pandas as pd
from collections import Counter

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "mabsa"
RESULTS_DIR = PROJECT_ROOT / "results" / "eda"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())
print("Results directory:", RESULTS_DIR.resolve())

Project root: D:\projects\J26-DS-349\component_1
Data directory: D:\projects\J26-DS-349\component_1\data\raw\mabsa
Results directory: D:\projects\J26-DS-349\component_1\results\eda


In [4]:
expected_files = []

for domain in ["phone", "laptop"]:
    for split in ["train", "dev", "test"]:
        path = DATA_DIR / domain / "en" / f"{split}.txt"
        expected_files.append(path)

for path in expected_files:
    print(path, "->", "FOUND" if path.exists() else "MISSING")

..\data\raw\mabsa\phone\en\train.txt -> FOUND
..\data\raw\mabsa\phone\en\dev.txt -> FOUND
..\data\raw\mabsa\phone\en\test.txt -> FOUND
..\data\raw\mabsa\laptop\en\train.txt -> FOUND
..\data\raw\mabsa\laptop\en\dev.txt -> FOUND
..\data\raw\mabsa\laptop\en\test.txt -> FOUND


In [5]:
sample_file = DATA_DIR / "phone" / "en" / "train.txt"

with open(sample_file, "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline().strip())
        print("-" * 100)

Bought this as a quick buy for my son the day he smashed his Huawei. Didn't want to buy anything too expensive for the same above reason. He is very happy with it. The resolution is amazing and it's super fast. The screen size is a perfect fit in the palm of the hand but he uses it popsocket for a better grip. Memory is huge which is a benefit too. Phone screen protection is on already and phone case cover is also included. Good bargain####[['happy with', 'Overall#Overall', 'positive'], ['resolution', 'Screen#Clarity', 'positive'], ['screen size', 'Screen#Size', 'positive'], ['Memory', 'Product Configuration#Operating Memory', 'positive'], ['Memory', 'Product Configuration#Operating Memory', 'positive'], ['Phone screen protection', 'Product Accessories#Cell Phone Film', 'positive'], ['phone case cover', 'Product Accessories#Phone Cases', 'positive'], ['bargain', 'Price#Price', 'positive']]
-------------------------------------------------------------------------------------------------

In [6]:
def parse_mabsa_line(line):
    line = line.strip()

    if not line:
        return None

    if "####" not in line:
        return {
            "text": line,
            "triplets": [],
            "parse_status": "no_separator"
        }

    text, label_text = line.split("####", 1)

    try:
        triplets = ast.literal_eval(label_text)
        status = "ok"
    except Exception:
        triplets = []
        status = "label_parse_error"

    return {
        "text": text.strip(),
        "triplets": triplets,
        "parse_status": status
    }

In [7]:
with open(sample_file, "r", encoding="utf-8") as f:
    first_line = f.readline()

parse_mabsa_line(first_line)

{'text': "Bought this as a quick buy for my son the day he smashed his Huawei. Didn't want to buy anything too expensive for the same above reason. He is very happy with it. The resolution is amazing and it's super fast. The screen size is a perfect fit in the palm of the hand but he uses it popsocket for a better grip. Memory is huge which is a benefit too. Phone screen protection is on already and phone case cover is also included. Good bargain",
 'triplets': [['happy with', 'Overall#Overall', 'positive'],
  ['resolution', 'Screen#Clarity', 'positive'],
  ['screen size', 'Screen#Size', 'positive'],
  ['Memory', 'Product Configuration#Operating Memory', 'positive'],
  ['Memory', 'Product Configuration#Operating Memory', 'positive'],
  ['Phone screen protection',
   'Product Accessories#Cell Phone Film',
   'positive'],
  ['phone case cover', 'Product Accessories#Phone Cases', 'positive'],
  ['bargain', 'Price#Price', 'positive']],
 'parse_status': 'ok'}

In [8]:
def load_split(domain, split):
    file_path = DATA_DIR / domain / "en" / f"{split}.txt"

    rows = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            parsed = parse_mabsa_line(line)

            if parsed is None:
                continue

            rows.append({
                "domain": domain,
                "split": split,
                "line_number": line_number,
                "text": parsed["text"],
                "triplets": parsed["triplets"],
                "parse_status": parsed["parse_status"],
            })

    return pd.DataFrame(rows)


In [9]:
dfs = []

for domain in ["phone", "laptop"]:
    for split in ["train", "dev", "test"]:
        dfs.append(load_split(domain, split))

df = pd.concat(dfs, ignore_index=True)

print("Combined dataset shape:", df.shape)

df.head()

Combined dataset shape: (4227, 6)


,domain,split,line_number,text,triplets,parse_status
0,phone,train,1,Bought this as a quick buy for my son the day ...,"[[happy with, Overall#Overall, positive], [res...",ok
1,phone,train,2,Camera quality not well. battery life not as p...,"[[Camera quality, Camera#General, negative], [...",ok
2,phone,train,3,Your browser does not support HTML5 video. ?Vf...,"[[browser, System#Software Compatibility, nega...",ok
3,phone,train,4,Received item as described. Delivered from ove...,"[[Received item, Logistics#Lost and Damaged, p...",ok
4,phone,train,5,Item as described. Fast delivery. Have purchas...,"[[delivery, Logistics#Speed, positive]]",ok


In [10]:
dataset_summary = (
    df.groupby(["domain", "split"])
      .size()
      .reset_index(name="num_sentences")
)

dataset_summary
dataset_summary.to_csv(
    RESULTS_DIR / "dataset_summary.csv",
    index=False
)

In [11]:
df["parse_status"].value_counts()

parse_status
ok    4227
Name: count, dtype: int64

In [12]:
df["num_triplets"] = df["triplets"].apply(len)

df.groupby("domain")["num_triplets"].describe()


,count,mean,std,min,25%,50%,75%,max
domain,,,,,,,,
laptop,2122.0,1.392554,0.766459,1.0,1.0,1.0,2.0,8.0
phone,2105.0,2.285036,1.836856,0.0,1.0,2.0,3.0,13.0


In [13]:
empty_labels = (
    df.assign(no_label=df["num_triplets"] == 0)
      .groupby("domain")["no_label"]
      .agg(["sum", "mean"])
)

empty_labels

,sum,mean
domain,,
laptop,0,0.000000
phone,327,0.155344


In [14]:
triplet_rows = []

for _, row in df.iterrows():
    for triplet in row["triplets"]:
        if len(triplet) != 3:
            continue

        aspect_term, aspect_category, sentiment = triplet

        triplet_rows.append({
            "domain": row["domain"],
            "split": row["split"],
            "text": row["text"],
            "aspect_term": aspect_term,
            "aspect_category": aspect_category,
            "sentiment": sentiment
        })

triplets_df = pd.DataFrame(triplet_rows)

print(triplets_df.shape)
triplets_df.head()

(7765, 6)


,domain,split,text,aspect_term,aspect_category,sentiment
0,phone,train,Bought this as a quick buy for my son the day ...,happy with,Overall#Overall,positive
1,phone,train,Bought this as a quick buy for my son the day ...,resolution,Screen#Clarity,positive
2,phone,train,Bought this as a quick buy for my son the day ...,screen size,Screen#Size,positive
3,phone,train,Bought this as a quick buy for my son the day ...,Memory,Product Configuration#Operating Memory,positive
4,phone,train,Bought this as a quick buy for my son the day ...,Memory,Product Configuration#Operating Memory,positive


In [15]:
phone_aspects = sorted(
    triplets_df.loc[
        triplets_df["domain"] == "phone",
        "aspect_category"
    ].dropna().unique()
)

laptop_aspects = sorted(
    triplets_df.loc[
        triplets_df["domain"] == "laptop",
        "aspect_category"
    ].dropna().unique()
)

print("PHONE ASPECT CATEGORIES:")
for x in phone_aspects:
    print(x)

print("\nLAPTOP ASPECT CATEGORIES:")
for x in laptop_aspects:
    print(x)

PHONE ASPECT CATEGORIES:
After-sales Service#Exchange/Warranty/Return
Appearance Design#Aesthetics General
Appearance Design#Color
Appearance Design#Exterior Design Material
Appearance Design#Fuselage Size
Appearance Design#Grip Feeling
Appearance Design#Thickness
Appearance Design#Weight
Appearance Design#Workmanship and Texture
Audio/Sound#Tone quality
Audio/Sound#Volume and Speaker
Battery/Longevity#Battery Capacity
Battery/Longevity#Battery Life
Battery/Longevity#Charging Method
Battery/Longevity#Charging Speed
Battery/Longevity#General
Battery/Longevity#Power Consumption Speed
Battery/Longevity#Standby Time
Branding/Marketing#Promotional Giveaways
Buyer Attitude#Loyalty
Buyer Attitude#Recommendable
Buyer Attitude#Repurchase and Churn Tendency
Buyer Attitude#Shopping Experiences
Buyer Attitude#Shopping Willingness
Camera#Fill light
Camera#Front Camera
Camera#General
Camera#Rear Camera
Ease of Use#Audience Groups
Ease of Use#Easy to Use
Intelligent Assistant#Intelligent Assistant Ge

In [16]:
sorted(triplets_df["sentiment"].dropna().unique())

sentiment_summary = (
    triplets_df.groupby(["domain", "sentiment"])
               .size()
               .reset_index(name="count")
)

sentiment_summary

,domain,sentiment,count
0,laptop,negative,921
1,laptop,neutral,177
2,laptop,positive,1857
3,phone,negative,1057
4,phone,neutral,198
5,phone,positive,3555


In [17]:
sentiment_summary.to_csv(
    RESULTS_DIR / "sentiment_summary.csv",
    index=False
)

In [18]:
aspect_summary = (
    triplets_df.groupby(["domain", "aspect_category"])
               .size()
               .reset_index(name="count")
               .sort_values(
                   ["domain", "count"],
                   ascending=[True, False]
               )
)

aspect_summary

,domain,aspect_category,count
48,laptop,LAPTOP#GENERAL,624
50,laptop,LAPTOP#OPERATION_PERFORMANCE,319
47,laptop,LAPTOP#DESIGN_FEATURES,243
52,laptop,LAPTOP#PRICE,116
53,laptop,LAPTOP#QUALITY,104
...,...,...,...
132,phone,Camera#Fill light,2
163,phone,Product Quality#Cleanliness,2
180,phone,Signal#Call Quality,2
141,phone,Logistics#Shipping Fee,1


In [19]:
aspect_summary.to_csv(
    RESULTS_DIR / "aspect_summary.csv",
    index=False
)

In [20]:
multi_aspect_summary = (
    df.assign(
        is_multi_aspect=df["num_triplets"] > 1
    )
    .groupby("domain")
    .agg(
        total_reviews=("text", "count"),
        multi_aspect_reviews=("is_multi_aspect", "sum"),
        multi_aspect_rate=("is_multi_aspect", "mean"),
    )
)

multi_aspect_summary

,total_reviews,multi_aspect_reviews,multi_aspect_rate
domain,,,
laptop,2122,592,0.278982
phone,2105,1340,0.636580


In [21]:
null_aspects = triplets_df[
    triplets_df["aspect_term"].astype(str).str.upper() == "NULL"
]

print("NULL aspect-term count:", len(null_aspects))
null_aspects.head(20)

NULL aspect-term count: 627


,domain,split,text,aspect_term,aspect_category,sentiment
4811,laptop,train,i bought this as a gift and was incredibly dis...,NULL,LAPTOP#GENERAL,negative
4815,laptop,train,the last straw happened recently when i put it...,NULL,LAPTOP#OPERATION_PERFORMANCE,negative
4827,laptop,train,it ' s so easy to travel with .,NULL,LAPTOP#PORTABILITY,positive
4829,laptop,train,overall i ' m happy with it for what i use it ...,NULL,LAPTOP#GENERAL,positive
4830,laptop,train,but it lost the coil whine roulette - - badly .,NULL,LAPTOP#DESIGN_FEATURES,negative
4835,laptop,train,it is far and away the best i ' ve ever had .,NULL,LAPTOP#GENERAL,positive
4851,laptop,train,that is just incredible .,NULL,LAPTOP#GENERAL,positive
4855,laptop,train,"pros : nice size , clear screen , quick on sta...",NULL,LAPTOP#DESIGN_FEATURES,positive
4857,laptop,train,"pros : nice size , clear screen , quick on sta...",NULL,LAPTOP#OPERATION_PERFORMANCE,positive
4858,laptop,train,"pros : nice size , clear screen , quick on sta...",NULL,LAPTOP#USABILITY,positive


In [25]:
print("=" * 60)
print("1. DATASET SUMMARY")
print("=" * 60)
print(dataset_summary.to_string(index=False))


print("\n" + "=" * 60)
print("2. PHONE ASPECT CATEGORIES")
print("=" * 60)

for aspect in phone_aspects:
    print(aspect)


print("\n" + "=" * 60)
print("3. LAPTOP ASPECT CATEGORIES")
print("=" * 60)

for aspect in laptop_aspects:
    print(aspect)


print("\n" + "=" * 60)
print("4. SENTIMENT LABELS")
print("=" * 60)

for label in sentiment_labels:
    print(label)


print("\n" + "=" * 60)
print("5. NULL ASPECT-TERM COUNT")
print("=" * 60)

print(len(null_aspects))


print("\n" + "=" * 60)
print("6. MULTI-ASPECT SUMMARY")
print("=" * 60)

print(multi_aspect_summary)

1. DATASET SUMMARY
domain split  num_sentences
laptop   dev            326
laptop  test            532
laptop train           1264
 phone   dev            307
 phone  test            522
 phone train           1276

2. PHONE ASPECT CATEGORIES
After-sales Service#Exchange/Warranty/Return
Appearance Design#Aesthetics General
Appearance Design#Color
Appearance Design#Exterior Design Material
Appearance Design#Fuselage Size
Appearance Design#Grip Feeling
Appearance Design#Thickness
Appearance Design#Weight
Appearance Design#Workmanship and Texture
Audio/Sound#Tone quality
Audio/Sound#Volume and Speaker
Battery/Longevity#Battery Capacity
Battery/Longevity#Battery Life
Battery/Longevity#Charging Method
Battery/Longevity#Charging Speed
Battery/Longevity#General
Battery/Longevity#Power Consumption Speed
Battery/Longevity#Standby Time
Branding/Marketing#Promotional Giveaways
Buyer Attitude#Loyalty
Buyer Attitude#Recommendable
Buyer Attitude#Repurchase and Churn Tendency
Buyer Attitude#Shopping 

NameError: name 'sentiment_labels' is not defined

In [26]:
null_summary = (
    triplets_df.assign(
        is_null=triplets_df["aspect_term"].astype(str).str.upper().eq("NULL")
    )
    .groupby("domain")
    .agg(
        total_triplets=("aspect_term", "count"),
        null_aspect_terms=("is_null", "sum"),
        null_rate=("is_null", "mean"),
    )
)

null_summary

,total_triplets,null_aspect_terms,null_rate
domain,,,
laptop,2955,627,0.212183
phone,4810,0,0.000000


In [27]:
null_summary.to_csv(
    RESULTS_DIR / "null_aspect_summary.csv",
    index=True
)

In [28]:
triplets_df[
    triplets_df["aspect_term"].astype(str).str.upper().eq("NULL")
][
    ["domain", "text", "aspect_category", "sentiment"]
].head(30)

,domain,text,aspect_category,sentiment
4811,laptop,i bought this as a gift and was incredibly dis...,LAPTOP#GENERAL,negative
4815,laptop,the last straw happened recently when i put it...,LAPTOP#OPERATION_PERFORMANCE,negative
4827,laptop,it ' s so easy to travel with .,LAPTOP#PORTABILITY,positive
4829,laptop,overall i ' m happy with it for what i use it ...,LAPTOP#GENERAL,positive
4830,laptop,but it lost the coil whine roulette - - badly .,LAPTOP#DESIGN_FEATURES,negative
4835,laptop,it is far and away the best i ' ve ever had .,LAPTOP#GENERAL,positive
4851,laptop,that is just incredible .,LAPTOP#GENERAL,positive
4855,laptop,"pros : nice size , clear screen , quick on sta...",LAPTOP#DESIGN_FEATURES,positive
4857,laptop,"pros : nice size , clear screen , quick on sta...",LAPTOP#OPERATION_PERFORMANCE,positive
4858,laptop,"pros : nice size , clear screen , quick on sta...",LAPTOP#USABILITY,positive


In [29]:
aspect_frequency = (
    triplets_df.groupby(["domain", "aspect_category"])
    .size()
    .reset_index(name="count")
    .sort_values(["domain", "count"], ascending=[True, False])
)

aspect_frequency

,domain,aspect_category,count
48,laptop,LAPTOP#GENERAL,624
50,laptop,LAPTOP#OPERATION_PERFORMANCE,319
47,laptop,LAPTOP#DESIGN_FEATURES,243
52,laptop,LAPTOP#PRICE,116
53,laptop,LAPTOP#QUALITY,104
...,...,...,...
132,phone,Camera#Fill light,2
163,phone,Product Quality#Cleanliness,2
180,phone,Signal#Call Quality,2
141,phone,Logistics#Shipping Fee,1


In [30]:
aspect_frequency[
    aspect_frequency["domain"] == "phone"
].head(30)

,domain,aspect_category,count
144,phone,Overall#Overall,910
142,phone,Logistics#Speed,402
140,phone,Logistics#Lost and Damaged,256
128,phone,Buyer Attitude#Recommendable,193
148,phone,Price#Price,150
173,phone,Seller Service#Attitude,149
134,phone,Camera#General,145
166,phone,Product Quality#General,123
120,phone,Battery/Longevity#Battery Life,105
177,phone,Seller Service#Timeliness of Seller Service,103


In [31]:
aspect_frequency[
    aspect_frequency["domain"] == "laptop"
].head(30)

,domain,aspect_category,count
48,laptop,LAPTOP#GENERAL,624
50,laptop,LAPTOP#OPERATION_PERFORMANCE,319
47,laptop,LAPTOP#DESIGN_FEATURES,243
52,laptop,LAPTOP#PRICE,116
53,laptop,LAPTOP#QUALITY,104
2,laptop,BATTERY#OPERATION_PERFORMANCE,76
39,laptop,KEYBOARD#GENERAL,75
14,laptop,DISPLAY#GENERAL,74
13,laptop,DISPLAY#DESIGN_FEATURES,67
54,laptop,LAPTOP#USABILITY,67


In [32]:
aspect_frequency.to_csv(
    RESULTS_DIR / "aspect_frequency.csv",
    index=False
)

In [33]:
phone_mapping = pd.DataFrame({
    "domain": "phone",
    "source_label": phone_aspects
})

laptop_mapping = pd.DataFrame({
    "domain": "laptop",
    "source_label": laptop_aspects
})

taxonomy_mapping = pd.concat(
    [phone_mapping, laptop_mapping],
    ignore_index=True
)

taxonomy_mapping["canonical_aspect"] = ""
taxonomy_mapping["semantic_group"] = ""
taxonomy_mapping["include"] = ""
taxonomy_mapping["mapping_status"] = "pending"
taxonomy_mapping["notes"] = ""

taxonomy_mapping.head()

,domain,source_label,canonical_aspect,semantic_group,include,mapping_status,notes
0,phone,After-sales Service#Exchange/Warranty/Return,,,,pending,
1,phone,Appearance Design#Aesthetics General,,,,pending,
2,phone,Appearance Design#Color,,,,pending,
3,phone,Appearance Design#Exterior Design Material,,,,pending,
4,phone,Appearance Design#Fuselage Size,,,,pending,


In [34]:
taxonomy_mapping.to_csv(
    PROJECT_ROOT / "taxonomy" / "mabsa_taxonomy_mapping_v0_1.csv",
    index=False
)

print("Mapping template saved.")

Mapping template saved.
